# R4 — Intensidad y excepciones de orden en VGG-16

Este notebook responde al comentario R4 del editor. La pregunta central es sencilla:

> ¿Cada imagen muestra más alteración al pasar de intensidad baja a media y de media a alta?

Para que ambas métricas tengan la misma dirección, se usa **sensibilidad angular = 1 − similitud coseno**. En las dos métricas, un valor mayor significa una alteración mayor.

El análisis reproduce Kendall's W y añade medidas más fáciles de interpretar: medianas, cambios entre intensidades, porcentaje de imágenes con el orden esperado y número de inversiones entre rotación de 30° y 60°.

In [ ]:
# En Google Colab, monta Drive y cambia CSV_PATH por la ruta real.
from google.colab import drive
drive.mount('/content/drive')

CSV_PATH = '/content/drive/MyDrive/embeddings_metricas_VGG16_TrashNet_completo_v2.csv'
OUTPUT_DIR = '/content/drive/MyDrive/Revision_R4_Intensity_VGG16'

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import friedmanchisquare, rankdata, wilcoxon

out = Path(OUTPUT_DIR)
out.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CSV_PATH)
print('Filas:', len(df))
print('Imágenes:', df['image_id'].nunique())
display(df.head())

assert len(df) == 2527 * 4 * 3 * 5
assert df.duplicated(['image_id', 'transformation', 'level', 'block']).sum() == 0
assert df[['cosine_similarity', 'relative_euclidean_distance']].isna().sum().sum() == 0
print('Auditoría del CSV: CORRECTA')

## Cómo leer el orden esperado

- Similitud coseno original: debería disminuir al aumentar la intensidad.
- Sensibilidad angular (`1 − coseno`): debería aumentar.
- Distancia euclidiana relativa: debería aumentar.

Por eso, en las tablas siguientes, el orden esperado siempre se escribe como:

`low < medium < high`

In [ ]:
LEVELS = ['low', 'medium', 'high']
BLOCKS = [f'Block{i}' for i in range(1, 6)]
TRANSFORMATIONS = ['brightness', 'contrast', 'rotation', 'zoom']
METRICS = {
    'cosine_similarity': 'Angular sensitivity (1 - cosine)',
    'relative_euclidean_distance': 'Relative Euclidean distance',
}

def sensitivity_frame(data, transformation, block, metric):
    sub = data[(data.transformation == transformation) & (data.block == block)]
    wide = sub.pivot(index=['image_id', 'class'], columns='level', values=metric)[LEVELS]
    return 1 - wide if metric == 'cosine_similarity' else wide

def rank_biserial(differences):
    d = np.asarray(differences, dtype=float)
    d = d[np.isfinite(d) & (d != 0)]
    ranks = rankdata(np.abs(d), method='average')
    return (ranks[d > 0].sum() - ranks[d < 0].sum()) / ranks.sum()

rows = []
for transformation in TRANSFORMATIONS:
    for metric, metric_label in METRICS.items():
        for block in BLOCKS:
            x = sensitivity_frame(df, transformation, block, metric)
            n = len(x)
            chi2, p = friedmanchisquare(x.low, x.medium, x.high)
            expected = (x.low < x.medium) & (x.medium < x.high)
            d_lm = x.medium - x.low
            d_mh = x.high - x.medium
            d_lh = x.high - x.low
            rows.append({
                'transformation': transformation,
                'metric': metric_label,
                'block': block,
                'n': n,
                'median_low': x.low.median(),
                'median_medium': x.medium.median(),
                'median_high': x.high.median(),
                'median_delta_low_medium': d_lm.median(),
                'median_delta_medium_high': d_mh.median(),
                'median_delta_low_high': d_lh.median(),
                'rank_biserial_low_medium': rank_biserial(d_lm),
                'rank_biserial_medium_high': rank_biserial(d_mh),
                'rank_biserial_low_high': rank_biserial(d_lh),
                'expected_order_n': int(expected.sum()),
                'expected_order_pct': 100 * expected.mean(),
                'inversion_medium_high_n': int((x.high <= x.medium).sum()),
                'inversion_medium_high_pct': 100 * (x.high <= x.medium).mean(),
                'inversion_low_high_n': int((x.high <= x.low).sum()),
                'inversion_low_high_pct': 100 * (x.high <= x.low).mean(),
                'friedman_chi2': chi2,
                'friedman_p': p,
                'kendall_w': chi2 / (n * 2),
            })

summary = pd.DataFrame(rows)
display(summary.head())

## Tabla 3 revisada

La tabla principal se concentra en los dos bloques señalados por el editor. No oculta la significancia; simplemente da prioridad a la magnitud y a las excepciones individuales.

In [ ]:
table3 = summary[(summary.transformation == 'rotation') & summary.block.isin(['Block3', 'Block5'])].copy()
table3 = table3[[
    'metric', 'block', 'n', 'median_low', 'median_medium', 'median_high',
    'median_delta_low_medium', 'median_delta_medium_high',
    'expected_order_n', 'expected_order_pct',
    'inversion_medium_high_n', 'inversion_medium_high_pct',
    'inversion_low_high_n', 'inversion_low_high_pct', 'kendall_w'
]]
display(table3.round(4))

table3.to_csv(out / '01_Table_3_rotation_intensity_ordering.csv', index=False)
summary.to_csv(out / '02_Table_S1_all_intensity_ordering.csv', index=False)

## Interpretación esencial

Una inversión `60° < 30°` no significa que la transformación de 60° sea débil en general. Significa que, para esa imagen concreta, la representación cambió un poco menos a 60° que a 30°. La comparación entre 10° y 60° continúa mostrando el aumento esperado en prácticamente todas las imágenes.

In [ ]:
rotation = summary[summary.transformation == 'rotation']
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
colors = {'Angular sensitivity (1 - cosine)': '#2F6B9A',
          'Relative Euclidean distance': '#D97732'}

for metric, color in colors.items():
    s = rotation[rotation.metric == metric].set_index('block').loc[BLOCKS]
    axes[0].plot(BLOCKS, s.expected_order_pct, marker='o', lw=2, color=color, label=metric)
    axes[1].plot(BLOCKS, s.inversion_medium_high_pct, marker='o', lw=2, color=color, label=metric)

axes[0].set_title('(a) Images following 10° < 30° < 60°')
axes[0].set_ylabel('Images (%)')
axes[0].set_ylim(78, 101.5)
axes[1].set_title('(b) Images less altered at 60° than 30°')
axes[1].set_ylabel('Images (%)')
axes[1].set_ylim(0, 20)
for ax in axes:
    ax.set_xlabel('VGG-16 block')
    ax.grid(axis='y', alpha=.3)
    ax.spines[['top', 'right']].set_visible(False)
    ax.legend(frameon=False, fontsize=8)

fig.suptitle('Rotation intensity ordering across individual TrashNet images', fontweight='bold')
fig.savefig(out / 'Figure_R4_rotation_intensity_exceptions.png', dpi=300, bbox_inches='tight')
plt.show()

## Resultado esperado

- Block3, sensibilidad angular: 91.06% siguió el orden completo; 8.94% mostró menos cambio a 60° que a 30°.
- Block5, sensibilidad angular: 87.69% siguió el orden completo; 12.11% mostró esa inversión.
- Block3, distancia relativa: 93.87% siguió el orden completo; 6.13% mostró la inversión.
- Block5, distancia relativa: 82.43% siguió el orden completo; 17.45% mostró la inversión.
- En Block5, solamente una imagen de 2,527 mostró menos alteración a 60° que a 10°.

Esto indica un aumento fuerte en el conjunto, pero no un orden perfectamente uniforme en cada imagen.